# AIMO3 TIR Baseline

Tool-Integrated Reasoning with vLLM + majority voting.

**Setup:**
- Model: QwQ-32B AWQ (`qwen-lm/qwq-32b/transformers/qwq-32b-awq/1`)
- Packages: installed offline from `animsamuelk/utils-aimo-3`
- Inference: vLLM with auto-detected tensor_parallel (1 or 2 GPUs)
- Strategy: N=16 samples with TIR, majority vote

In [ ]:
import subprocess
import sys

# ── GPU detection ─────────────────────────────────────────────────────────────
_has_nvidia = (
    subprocess.run(["which", "nvidia-smi"], capture_output=True).returncode == 0
)

if _has_nvidia:
    _smi = subprocess.check_output(
        [
            "nvidia-smi",
            "--query-gpu=name,memory.total,compute_cap",
            "--format=csv,noheader",
        ],
        text=True,
    ).strip()
    print(f"GPUs:\n{_smi}", flush=True)
    _gpu_count = _smi.count("\n") + 1
    # Parse compute capability of first GPU (e.g. "8.0" → 80, "7.5" → 75, "6.0" → 60)
    _cap_str = _smi.split("\n")[0].split(",")[-1].strip()  # e.g. "7.5"
    _gpu_cap = int(_cap_str.replace(".", ""))  # e.g. 75
else:
    _gpu_count = 0
    _gpu_cap = 0
    print("No nvidia-smi found — CPU only", flush=True)

MODEL_PATH = "/kaggle/input/models/qwen-lm/qwq-32b/transformers/qwq-32b-awq/1"
N_SAMPLES = 16
TENSOR_PARALLEL = max(1, _gpu_count)
MAX_TIR_STEPS = 6
VLLM_PORT = 8080
VLLM_HOST = f"http://localhost:{VLLM_PORT}/v1"
GPU_MEMORY_UTIL = 0.90
MAX_MODEL_LEN = 16384 if TENSOR_PARALLEL >= 2 else 8192
# Per-step token limit: sized so MAX_TIR_STEPS fit within MAX_MODEL_LEN.
MAX_TOKENS = (MAX_MODEL_LEN - 2000) // MAX_TIR_STEPS

# AWQ requires compute capability >= 7.5 (75). On P100 (6.0/60), vLLM will fail.
print(f"GPU count: {_gpu_count}, compute_cap: {_gpu_cap}", flush=True)
print(
    f"N_SAMPLES={N_SAMPLES}, TP={TENSOR_PARALLEL}, max_model_len={MAX_MODEL_LEN}, max_tokens/step={MAX_TOKENS}"
)

# Write placeholder submission.parquet immediately so output file exists
# even if vLLM fails to start or serve() blocks in a non-competition run.
import polars as pl

pl.DataFrame({"id": ["0"], "answer": [42]}).write_parquet(
    "/kaggle/working/submission.parquet"
)
print("Placeholder submission.parquet written.", flush=True)

In [ ]:
# ── Install packages from offline dataset (no internet in competition run) ───
# animsamuelk/utils-aimo-3: vLLM 0.16.0 + Ray 2.54.0 + grpcio 1.78.0 +
#   protobuf 6.33.5 + outlines_core 0.2.11 + all deps (185 wheels, cp312, March 2026)
import os
import subprocess

DEPS_DIR = "/kaggle/input/utils-aimo-3"

_wheels = (
    sorted(f for f in os.listdir(DEPS_DIR) if f.endswith(".whl"))
    if os.path.isdir(DEPS_DIR)
    else []
)
print(f"DEPS_DIR has {len(_wheels)} wheels", flush=True)

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-index",
        "--find-links",
        DEPS_DIR,
        "vllm",
        "openai",
        "polars",
    ],
    check=True,
)
print("Packages installed", flush=True)

In [ ]:
# ── Start vLLM inference server ──────────────────────────────────────────────
import subprocess
import sys
import time
import urllib.request

VLLM_LOG = "/kaggle/working/vllm.log"

# Fatal error patterns: vLLM process is alive but will never serve
_FATAL_PATTERNS = [
    "is not supported for the current GPU",  # AWQ/GPTQ incompatible GPU (e.g. P100 sm_60)
    "ValidationError",  # pydantic config validation failure
    "CUDA out of memory",  # OOM: won't recover
    "No CUDA GPUs are available",  # No GPU at all
]

vllm_proc = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "vllm.entrypoints.openai.api_server",
        "--model",
        MODEL_PATH,
        "--port",
        str(VLLM_PORT),
        "--tensor-parallel-size",
        str(TENSOR_PARALLEL),
        "--gpu-memory-utilization",
        str(GPU_MEMORY_UTIL),
        "--max-model-len",
        str(MAX_MODEL_LEN),
        "--max-num-seqs",
        "32",
        "--trust-remote-code",
        "--disable-log-requests",
    ],
    stdout=open(VLLM_LOG, "w"),
    stderr=subprocess.STDOUT,
)

print("Waiting for vLLM server...", flush=True)
VLLM_STARTED = False
for i in range(120):
    # Exit immediately if the process crashed
    if vllm_proc.poll() is not None:
        print(
            f"vLLM process exited (code={vllm_proc.returncode}) after {i * 5}s",
            flush=True,
        )
        break
    try:
        urllib.request.urlopen(f"http://localhost:{VLLM_PORT}/health", timeout=2)
        print(f"vLLM ready after {i * 5}s", flush=True)
        VLLM_STARTED = True
        break
    except Exception:
        # Check for fatal errors in the log — process stays alive but will never serve
        if i >= 2:  # give vLLM a few seconds to write something meaningful
            try:
                log_text = open(VLLM_LOG).read()
                for pat in _FATAL_PATTERNS:
                    if pat in log_text:
                        print(
                            f"vLLM fatal error detected ({pat!r}) — exiting early",
                            flush=True,
                        )
                        vllm_proc.terminate()
                        VLLM_STARTED = False
                        break
                else:
                    time.sleep(5)
                    if i % 6 == 5:
                        lines = open(VLLM_LOG).readlines()
                        print(f"[{i * 5}s] {lines[-1].strip()}", flush=True)
                    continue
                break  # fatal pattern found
            except Exception:
                time.sleep(5)
                continue
        else:
            time.sleep(5)

if not VLLM_STARTED:
    print(
        "WARNING: vLLM server failed to start — fallback predict() will return 42.",
        flush=True,
    )
    try:
        print("".join(open(VLLM_LOG).readlines()[-10:]), flush=True)
    except Exception:
        pass

In [ ]:
# ── TIR Solver ──────────────────────────────────────────────────────────────
import os
import re
import subprocess
import sys
import tempfile
from collections import Counter

from openai import OpenAI

SYSTEM_PROMPT = """\
You are an expert math competition solver. Solve the problem step by step.

When you need to compute something, write Python code in a ```python block.
You will see the output and can continue reasoning.

End your response with \\boxed{N} where N is your final integer answer (0-99999).
"""

_llm = OpenAI(base_url=VLLM_HOST, api_key="local")


def execute_python(code: str, timeout: int = 15) -> str:
    for pattern in [
        r"\bos\b",
        r"\bsubprocess\b",
        r"\bopen\b",
        r"__import__",
        r"socket",
        r"urllib",
        r"requests",
    ]:
        if re.search(pattern, code):
            return "[blocked: unsafe code]"
    with tempfile.NamedTemporaryFile(mode="w", suffix=".py", delete=False) as f:
        f.write("from math import *\nimport sympy\n" + code)
        tmp = f.name
    try:
        r = subprocess.run(
            [sys.executable, tmp], capture_output=True, text=True, timeout=timeout
        )
        out = r.stdout
        if r.returncode != 0 and r.stderr:
            out += f"\n[err]: {r.stderr[:300]}"
        return (out.strip() or "(no output)")[:2000]
    except subprocess.TimeoutExpired:
        return f"[timeout after {timeout}s]"
    finally:
        os.unlink(tmp)


def extract_code_blocks(text: str) -> list:
    return re.findall(r"```python\n(.*?)```", text, re.DOTALL)


def extract_answer(text: str):
    for m in re.finditer(r"\\boxed\{(\d+)\}", text):
        v = int(m.group(1))
        if 0 <= v <= 99999:
            return v
    for m in re.finditer(
        r"(?:answer\s+is|answer:|final\s+answer[:\s]+|=\s*)(\d+)", text, re.IGNORECASE
    ):
        v = int(m.group(1))
        if 0 <= v <= 99999:
            return v
    return None


def _chat(messages: list, temperature: float) -> str:
    r = _llm.chat.completions.create(
        model=MODEL_PATH,
        messages=messages,
        temperature=temperature,
        max_tokens=MAX_TOKENS,
    )
    return r.choices[0].message.content or ""


def solve_with_voting(problem: str, n: int = N_SAMPLES) -> int:
    answers = []
    for i in range(n):
        temp = 0.0 if i == 0 else 0.7
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": problem},
        ]
        for _ in range(MAX_TIR_STEPS):
            reply = _chat(messages, temperature=temp)
            messages.append({"role": "assistant", "content": reply})
            code_blocks = extract_code_blocks(reply)
            if code_blocks:
                out = execute_python(code_blocks[-1])
                messages.append({"role": "user", "content": f"Code output:\n{out}"})
            if not code_blocks:
                if extract_answer(reply) is not None:
                    break
                messages.append(
                    {"role": "user", "content": "Give your final answer as \\boxed{N}."}
                )
            elif extract_answer(reply) is not None:
                break
        all_text = "\n".join(m["content"] for m in messages if m["role"] == "assistant")
        ans = extract_answer(all_text)
        if ans is not None:
            answers.append(ans)
    return Counter(answers).most_common(1)[0][0] if answers else 0


print("Solver ready", flush=True)

In [ ]:
# ── Kaggle gateway integration ───────────────────────────────────────────────
import sys

import polars as pl

sys.path.insert(
    0, "/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3"
)
import kaggle_evaluation.aimo_3_inference_server as aimo3


def predict(df: pl.DataFrame) -> pl.DataFrame:
    """Called once per problem by the competition gateway."""
    row = df.row(0, named=True)
    problem_id = row["id"]
    problem = row["problem"]
    print(f"[{problem_id}] solving...", flush=True)
    if not VLLM_STARTED:
        # vLLM not available (incompatible GPU in regular kernel run)
        answer = 42
    else:
        answer = solve_with_voting(problem, n=N_SAMPLES)
    print(f"[{problem_id}] answer={answer}", flush=True)
    return pl.DataFrame({"id": [problem_id], "answer": [answer]})


inference_server = aimo3.AIMO3InferenceServer(predict)
inference_server.serve()